In [0]:
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
%sql
SELECT *,
row_number() over (partition by CODIGO_ANONIMIZADO order by TO_DATE(FECHA_ATENCION) DESC) as rn
FROM hta_sis.silver_hta

In [0]:
%sql
CREATE OR REPLACE VIEW vw_hd_hta AS
SELECT
    *
FROM hta_sis.silver_hta;

In [0]:
%sql
SELECT *
FROM vw_hd_hta

In [0]:
%sql
SELECT COUNT(*)
FROM hta_sis.silver_hta;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_hd_universo_hta AS
SELECT 
  vhhta.CODIGO_ANONIMIZADO,
  vhhta.SEXO,
  vhhta.ID_REGISTRO_REL AS hta_id_registro_rel,
  TO_DATE(vhhta.FECHA_ATENCION, 'yyyy-MM-dd') AS FECHA_ATENCION,  -- Aplicamos TO_DATE aquí
  vhhta.EDAD,
  vhhta.NIVEL_EESS,
  vhhta.CODIGO_SERV_PRESTACIONAL,
  vhhta.SERVICIO_PRESTACIONAL,
  vhhta.DIAS_HOSP,
  vhhta.PRES_ART_SISTOLICA,
  vhhta.PRES_ART_DIASTOLICA,
  vhhta.TIPO_PERSONAL_SALUD,
  vhhta.FECATE_POST_FECFED,
  vhhta.ES_CAPITA,
  vhhta.DESTINO_ASEGURADO,
  vhhta.PERIODO_ATENCION,
  sd.TIPO_DIAGNOSTICO,
  snd.nombre_diagnostico,
  
  LAG(TO_DATE(vhhta.FECHA_ATENCION, 'yyyy-MM-dd')) OVER (  -- Aplicamos TO_DATE en LAG
    PARTITION BY vhhta.CODIGO_ANONIMIZADO
    ORDER BY TO_DATE(vhhta.FECHA_ATENCION, 'yyyy-MM-dd')  -- Aseguramos que ORDER BY sea con TO_DATE
  ) AS FECHA_ATENCION_ANTERIOR,

  LAG(vhhta.PRES_ART_SISTOLICA) OVER (
    PARTITION BY vhhta.CODIGO_ANONIMIZADO
    ORDER BY TO_DATE(vhhta.FECHA_ATENCION, 'yyyy-MM-dd')  -- Ordenamos con TO_DATE
  ) AS SIST_ANTERIOR,

  LAG(vhhta.PRES_ART_DIASTOLICA) OVER (
    PARTITION BY vhhta.CODIGO_ANONIMIZADO
    ORDER BY TO_DATE(vhhta.FECHA_ATENCION, 'yyyy-MM-dd')  -- Ordenamos con TO_DATE
  ) AS DIAST_ANTERIOR,

  LAG(vhhta.DESTINO_ASEGURADO) OVER (
    PARTITION BY vhhta.CODIGO_ANONIMIZADO
    ORDER BY TO_DATE(vhhta.FECHA_ATENCION, 'yyyy-MM-dd')  -- Ordenamos con TO_DATE
  ) AS DESTINO_ANTERIOR,

  (TO_DATE(vhhta.FECHA_ATENCION, 'yyyy-MM-dd') 
   - LAG(TO_DATE(vhhta.FECHA_ATENCION, 'yyyy-MM-dd')) OVER (
      PARTITION BY vhhta.CODIGO_ANONIMIZADO
      ORDER BY TO_DATE(vhhta.FECHA_ATENCION, 'yyyy-MM-dd')  -- Aseguramos que la resta de fechas se haga correctamente
   )
  ) AS DIAS_DESDE_ANTERIOR

FROM vw_hd_hta AS vhhta
INNER JOIN hta_sis.silver_diag sd
  ON vhhta.ID_REGISTRO_REL = sd.ID_REGISTRO_REL
INNER JOIN hta_sis.silver_nam_diag snd
  ON sd.CODDIA = snd.coddia;

In [0]:
df_temp_hd_universal = spark.sql("""
SELECT *
FROM vw_hd_universo_hta """)

In [0]:
criterio_partition = Window.partitionBy(
    F.col("CODIGO_ANONIMIZADO"), F.col("FECHA_ATENCION")
).orderBy(F.to_date(F.col("FECHA_ATENCION"),"yyyyMMdd").desc())

df_temp_hd_universal = df_temp_hd_universal.withColumn("rank", F.row_number().over(criterio_partition))


df_temp_hd_universal = df_temp_hd_universal.filter(F.col("rank") == 1)

df_hd_universal = df_temp_hd_universal.drop("rank","DIAS_DESDE_ANTERIOR")

In [0]:
display(df_hd_universal)

In [0]:
target_path = "/Volumes/workspace/hta_sis/silver/hd_universal_hta"

if DeltaTable.isDeltaTable(spark, target_path):
    delta_table = DeltaTable.forPath(spark, target_path)
    delta_table.alias("t").merge(
        df_hd_universal.alias("s"),
        "t.ID_REGISTRO_REL = s.ID_REGISTRO_REL"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    df_hd_universal.write.format("delta").mode("overwrite").save(target_path)